# Etappe 11: Finale dag Afdaling (Excentrische Spiervermoeidheid)

**Activiteit:** Stubaier Höhenweg, etappe 11 (22 september 2026)

**Vraag:** Zakt mijn daalsnelheid (VDM - Vertical Descent Meters per uur) in gedurende de lange afdaling naar Fulpmes als gevolg van opgestapelde excentrische spiervermoeidheid in de bovenbenen?

**Methode:**
1. We isoleren de datapunten waarbij de hoogte daalt (excentrische belasting).
2. We berekenen de VDM (daalsnelheid in m/uur) met een voortschrijdend gemiddelde (rolling average van 10 minuten) om het ruwe GPS-signaal glad te strijken.
3. We plotten de daalsnelheid over de verstreken tijd, samen met het hoogteprofiel, om het efficiëntieverlies in de spieren ("Decay") zichtbaar te maken.

**Resultaat:** een afbeelding in de map `output/`.

## Installatie van de benodigde pakketten

De pakketten staan in `requirements.txt`. Deze cel hoeft alleen de eerste keer na het opstarten van het cluster te draaien.

In [0]:
%pip install -q -r ../requirements.txt

## Imports

In [0]:
from utils.berekeningen import bereken_daal_vermoeidheid
from utils.config import ANALYSE, PADEN
from utils.grafieken import teken_daal_profiel
from utils.inlezen import bereken_verstreken_tijd, lees_fit_bestand, schoon_fit_data
from utils.visualisatie import exporteer_afbeelding

## Instellingen

In [0]:
ETAPPE_NR = 11
FIT_BESTAND = PADEN.fit_map / "20260922_stubaier_hohenweg_etappe_11.fit"
WINDOW_MINUTEN = 10  # Voor het voortschrijdend gemiddelde

## Data inlezen

In [0]:
ruwe_data = lees_fit_bestand(FIT_BESTAND)
df = schoon_fit_data(ruwe_data, cadans_factor=ANALYSE.wandel_cadans_factor)
df = bereken_verstreken_tijd(df)

## Berekening: Isolatie van de klim en het steilste segment

In [0]:
df_dalen, statistieken = bereken_daal_vermoeidheid(df, window_minuten=WINDOW_MINUTEN)

print("--- Resultaten Afdaling ---")
print(f"Maximale daalsnelheid (VDM): {statistieken['max_vdm']:.0f} m/u")
print(f"Daalsnelheid start (eerste 20%): {statistieken['vdm_start']:.0f} m/u")
print(f"Daalsnelheid einde (laatste 20%): {statistieken['vdm_einde']:.0f} m/u")
print(f"Descent Decay (Snelheidsverlies): {statistieken['decay_percentage']:.1f}%")

## Grafiek

In [0]:
fig = teken_daal_profiel(df_dalen, statistieken, etappe_nr=ETAPPE_NR)

## Exporteren

In [0]:
exporteer_afbeelding(fig, bestandsnaam="20260922_final_descent_decay", formaat="liggend")